# Examine Relationship Between Combine Results And Separation During Go/Post/Corner Routes

**<span style="color: gold;">PROJECT</span>**

WR/DB Separation Analysis

**<span style="color: gold;">GOAL OF THIS NOTEBOOK</span>**

We have previously shown that some of our engineered features derived from NFL Combine player-tracking data have significant associations with a WRs ability to create separation on bursting routes.

We now want to test if that information tells us more than what general Combine results (40-yard dash time, 3 Cone drill time, etc.) tell us. That is, if coaches could learn just as much from simply looking at general Combine results, then they don't need our derived metrics. 

Therefore, we will test here if there are any significant relationships between general Combine metrics and a WRs ability to create space during bursts in go/post/corner routes. We hypothesize that no Combine features will have any predictive power on a WRs ability to create space.

Predictors: General Combine Results

Outcome: Separation At Cut During Hitch/In/Out Routes

**<span style="color: gold;">FINDINGS</span>**

*ngs_athleticism_score* also shows a statistically significant relationship with *avg_go_separation_generated*.

**<span style="color: gold;">NOTES</span>**

In this analysis, we will be derive separation metrics that use WR player-tracking data and player-tracking data for their defender. Due to our need for adequate defender data, the 2027 Big Data Bowl data will not suffice. Therefore, we will use the 2026 Big Data Bowl data, which means that we can only examine 2023 rookies.


<br><br/>
#### *Initialize Environment*

In [ ]:
import sys
from pathlib import Path

# Identify Project Root
PROJECT_ROOT = next(
    parent
    for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "pyproject.toml").exists()
)
sys.path.insert(0, str(PROJECT_ROOT))

# Import Shared Project Paths
from utils.paths import (
    CLEAN_DATA_DIR,
    BDB_2026_BEFORE_PASS_DIR,
    SEPARATION_DATA_DIR,
)

import polars as pl 
import pandas as pd 
import numpy as np
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "png"
from datetime import datetime

import statsmodels.api as sm

# Import schema_map For 2027 BDB Data
import sys
from pathlib import Path
# Identify Project Root

PROJECT_ROOT = next(

    parent

    for parent in [Path.cwd(), *Path.cwd().parents]

    if (parent / "pyproject.toml").exists()

)



# Import Shared Data Schemas

sys.path.insert(0, str(PROJECT_ROOT))

from utils.schema import schema_map

<br><br/>
## Load Data
***

#### Set Parameters For Analysis

In [ ]:
# =======================================
# Import Parameters That Define What 
# Data We'll Examine In This Project
# =======================================
param_path = PROJECT_ROOT / "05_WR_Separation_Analysis" / "Data_Pull"
sys.path.append(str(param_path))
import Data_Parameters as params

In [ ]:
# =======================================
# Specify Routes of Interest
# =======================================

ROUTES_OF_INTEREST = ["GO", "CORNER", "POST"]

<br><br/>
#### Play and Player Level Details

In [ ]:
# ===========================================
# Player-Time Level WR/DB Player-Tracking & Play Data
# ===========================================

play_player_details_df = pl.read_csv(str(SEPARATION_DATA_DIR / "00_2023_Rookie_Player_Time_Level_Play_Details_Data.csv"))

print("Play/Player Details Data:")
print(f"    - Number of Unique WRs: {play_player_details_df.filter(pl.col("player_position") == "WR").select("nfl_id").n_unique()}")
print(f"    - Number of WR Player-Plays: {play_player_details_df.filter(pl.col("player_position") == "WR").select("player_play_id").n_unique()}")
play_player_details_df

<br><br/>
#### WR Separation Data From *01_BDB26_WR_Player_Tracking.ipynb*

In [ ]:
# =========================================
# WR Separation Data Among 2023 Rookies
# =========================================

wr_separation_df = pl.read_csv(str(SEPARATION_DATA_DIR / "01_2023_Rookie_Player_Time_Level_Separation_Data.csv"))

# -----------------------------------------
# Identify Players Of Interest
# -----------------------------------------

players_of_interest = (
    wr_separation_df
    .select("wr_nfl_id")
    .unique()
    .to_series()
    .to_list()
)

print("Separation Data:")
print(f"    - Dimensions: {wr_separation_df.shape}")
print(f"    - Number of Rookie WRs: {wr_separation_df.select("wr_nfl_id").n_unique()}")
print(f"    - Number of Player-Plays: {wr_separation_df.select("player_play_id").n_unique()}")
wr_separation_df

<br><br/>
#### Combine Results Data

In [ ]:
# =======================================
# Combine Results Data
# =======================================

combine_results_df = (
    pl.read_csv(
        str(CLEAN_DATA_DIR / "combine_results.csv"), 
        schema_overrides = schema_map["combine_results"]
    )
)

combine_features_df = (
    pl.read_csv(str(SEPARATION_DATA_DIR / "03_WR_Combine_Features_Data.csv"))
    .select("nfl_id", "accel_score")
)

combine_results_df = (
    combine_results_df
    .join(
        combine_features_df, 
        on = "nfl_id", 
        how = "left"
    )
)

print("Combine Results Data:")
print(f"    - Dimensions: {combine_results_df.shape}")
print(f"    - Number of Players: {combine_results_df.select("nfl_id").n_unique()}")
combine_results_df

<br><br/>
## Data Cleaning
***

#### Filter Data Based On Inclusion Criteria

In [ ]:
# ===============================
# Number of WRs In WR Separation Data
# ===============================
wr_player_plays = wr_separation_df.select("player_play_id").unique().to_series().to_list()
print("WRs In WR Separation Data:")
print(f"    - Number of Unique WR Player-Plays: {len(wr_player_plays)}")

In [ ]:
# ===============================
# How Many of Those Player-Plays Do We Have Route Data For?
# ===============================

# -------------------------------
# For BDB2027 Route Data, Use route_ran
# -------------------------------

bdb2027_extra_route_data = (
    play_player_details_df
    .filter( 
        pl.col("route_ran").is_not_null()
    )
    .group_by("player_play_id")
    .agg(
        route_ran = pl.col("route_ran").first()
    )
)

# -------------------------------
# For BDB2026 Route Data For Targeted 
# Receivers, Use route_of_targetered_receiver
# -------------------------------

target_routes = (
    play_player_details_df
    .filter(
        pl.col("target"), 
        pl.col("route_of_targeted_receiver").is_not_null()
    )
    .group_by("player_play_id")
    .agg(
        route_of_targeted_receiver = pl.col("route_of_targeted_receiver").first()
    )
)

# -------------------------------
# Target Data
# -------------------------------

target_data = (
    play_player_details_df
    .select("player_play_id", "target")
    .unique()
)

# -------------------------------
# Add Routes To WR Separation Data
# -------------------------------

wr_separation_df = (
    wr_separation_df
    # Add target data
    .join(
        target_data, 
        on = "player_play_id", 
        how = "inner"
    )
    # Add non-targeted routes
    .join(
        bdb2027_extra_route_data, 
        on = "player_play_id", 
        how = "left"
    )
    # Add targeted routes
    .join(
        target_routes, 
        on = "player_play_id", 
        how = "left"
    )
    # Create route variable
    .with_columns(
        route = (
            # If target and has route_of_targeted_receiver data, then use that
            pl.when((pl.col("target") == True) & (pl.col("route_of_targeted_receiver").is_not_null()))
              .then(pl.col("route_of_targeted_receiver"))
            # Otherwise, use route_ran
              .otherwise(pl.col("route_ran"))
        )
    )
)

# -----------------------------
# Get Rid Of Player-Plays With No Route Data
# -----------------------------

wr_separation_df = (
    wr_separation_df
    .filter(pl.col("route").is_not_null())
)

# -----------------------------
# Only Consider Targeted Receivers
# -----------------------------

wr_separation_df = (
    wr_separation_df.filter("target")
)

# -----------------------------
# Only Consider These Route Types
# -----------------------------

wr_separation_df = (
    wr_separation_df
    .filter(pl.col("route").is_in(ROUTES_OF_INTEREST))
)

# -----------------------------
# Only Consider Man Coverage
# -----------------------------

wr_separation_df = (
    wr_separation_df
    # -------------------------------
    # Add team_coverage_man_zone
    # -------------------------------
    .join(
        # Get the coverage for each play
        play_player_details_df
        .select("game_play_id", "team_coverage_man_zone")
        # Add .unique() to make sure we don't have duplicate rows!
        .unique(), 
        on = "game_play_id",
        how = "inner"
    )
    # -------------------------------
    # Filter To Man Coverage
    # -------------------------------
    .filter(pl.col("team_coverage_man_zone") == "MAN_COVERAGE")
)


print(f"Updated Assignments Data:")
print(f"    - Number of WR Player-Plays: {wr_separation_df.select("player_play_id").n_unique()}")
print(f"    - Number of WRs: {wr_separation_df.select("wr_nfl_id").n_unique()}")
print(f"    - Number of Unique Plays: {wr_separation_df.select("game_play_id").n_unique()}")
wr_separation_df

<br><br/>
#### Add *absolute_yardline_number* To WR Separation Data

In [ ]:
# ===============================================
# Add absolute_yardline_number To wr_separation_df
# ===============================================

data_dir = Path(str(BDB_2026_BEFORE_PASS_DIR))

# ------------------------------------
# Load and Combine 18 Before Pass Datasets Into 1
# ------------------------------------

absolute_yardline_number_df = (
    pl.concat(
        [
            pl.scan_csv(data_dir / f"input_2023_w{i:02d}.csv")
            .with_columns(pl.lit(i).alias("week"))
            for i in range(1, 19)
        ],
        how="vertical"
    )
    .collect()
)

# ------------------------------------
# Add Game-Play and Player-Play ID
# ------------------------------------

absolute_yardline_number_df = (
    absolute_yardline_number_df
    .with_columns(game_play_id = pl.col("game_id").cast(pl.String) + "_" + pl.col("play_id").cast(pl.String))
    .group_by("game_play_id")
    .agg(absolute_yardline_number = pl.col("absolute_yardline_number").first())
)

absolute_yardline_number_df

In [ ]:
wr_separation_df = (
    wr_separation_df
    .join(
        absolute_yardline_number_df, 
        on = "game_play_id", 
        how = "inner"
    )
)

wr_separation_df

<br><br/>
#### Create Metrics For Separation In Routes

In [ ]:
# ==============================================================================
# Extract In-Game Acceleration and Separation for Go Routes
# ==============================================================================

# ------------------------------------------------------------------------------
# 1. Calculate Downfield Kinematics and Filter for Vertical Push
# ------------------------------------------------------------------------------
go_kinematics_df = (
    wr_separation_df
    .sort(["player_play_id", "frame_id"])
    .with_columns(
        # Calculate raw directional velocities (dt = 0.1s at 10Hz)
        vx = (pl.col("wr_x") - pl.col("wr_x").shift(1).over("player_play_id")) / 0.1,
        vy = (pl.col("wr_y") - pl.col("wr_y").shift(1).over("player_play_id")) / 0.1,
    )
    .with_columns(
        # Speed magnitude
        s = (pl.col("vx")**2 + pl.col("vy")**2).sqrt()
    )
    .with_columns(
        # True linear acceleration
        linear_accel = (pl.col("s") - pl.col("s").shift(1).over("player_play_id")) / 0.1
    )
    .with_columns(
        # Isolate true downfield velocity based on play direction
        v_downfield = pl.when(pl.col("play_direction") == "right")
                      .then(pl.col("vx"))
                      .otherwise(-pl.col("vx")),
        # Calculate relative distance from the Line of Scrimmage (LOS)
        # Assumes absolute distance downfield from where the play started
        yards_midfield = pl.when(pl.col("play_direction") == "right")
                         .then(pl.col("wr_x") - pl.col("absolute_yardline_number"))
                         .otherwise(pl.col("absolute_yardline_number") - pl.col("wr_x"))
    )
)

# ------------------------------------------------------------------------------
# 2. Isolate Key Frames: The Snap (Release) and Peak Downfield Burst
# ------------------------------------------------------------------------------
# For Go routes, we want to find the frame where they hit their absolute highest 
# acceleration gear *after* they've advanced at least 5 yards past the LOS. 
# This prevents capturing short hand-fights at the line and focuses on deep vertical burn.
go_burst_frames_df = (
    go_kinematics_df
    .filter(
        (pl.col("linear_accel").is_not_null()) & 
        (pl.col("yards_midfield") > 5.0) # Clear the initial line-of-scrimmage jam
    )
    .group_by("player_play_id")
    .agg([
        # Find the frame where the WR hits maximum acceleration downfield
        pl.col("frame_id").filter(pl.col("linear_accel") == pl.col("linear_accel").max()).first().alias("peak_burst_frame_id"),
        # Also capture the snap/start frame of the play for baseline release tracking
        pl.col("frame_id").min().alias("snap_frame_id")
    ])
)

# ------------------------------------------------------------------------------
# 3. Extract Separation at Release and Peak Vertical Burst
# ------------------------------------------------------------------------------
final_go_separation_df = (
    go_kinematics_df
    .join(go_burst_frames_df, on="player_play_id", how="inner")
    .filter(
        (pl.col("frame_id") == pl.col("snap_frame_id")) |                          # At the line
        (pl.col("frame_id") == (pl.col("snap_frame_id") + 10)) |                    # 1.0s after snap (Release Separation)
        (pl.col("frame_id") == pl.col("peak_burst_frame_id")) |                    # Moment of peak deep acceleration
        (pl.col("frame_id") == (pl.col("peak_burst_frame_id") + 5))                 # 0.5s after peak acceleration
    )
    .group_by("player_play_id")
    .agg([
        pl.col("wr_nfl_id").first(),
        pl.col("wr_name").first(),
        
        # 1. Initial Release Separation: How well they stack the CB in the first 10 frames (1 second)
        pl.col("wr_cb_distance").filter(pl.col("frame_id") == (pl.col("snap_frame_id") + 10)).first().alias("separation_at_release"),
        
        # 2. Deep Burst Separation: Distance to CB the exact moment they turn on their highest gear
        pl.col("wr_cb_distance").filter(pl.col("frame_id") == pl.col("peak_burst_frame_id")).first().alias("separation_at_peak_burst"),
        
        # 3. Post-Burst Separation: Did the acceleration actually burn the CB 0.5 seconds later?
        pl.col("wr_cb_distance").filter(pl.col("frame_id") == (pl.col("peak_burst_frame_id") + 5)).first().alias("separation_post_burst")
    ])
    .with_columns(
        # The ultimate metric: Net separation generated *by* the downfield burst
        go_separation_generated = pl.col("separation_post_burst") - pl.col("separation_at_peak_burst")
    )
)

# ------------------------------------------------------------------------------
# 4. Roll up to Player Averages
# ------------------------------------------------------------------------------
player_go_metrics_df = (
    final_go_separation_df
    .group_by("wr_nfl_id")
    .agg([
        pl.col("wr_name").first(),
        pl.col("separation_at_release").mean().alias("avg_release_separation"),
        pl.col("separation_at_peak_burst").mean().alias("avg_sep_at_burst_point"),
        pl.col("go_separation_generated").mean().alias("avg_go_separation_generated")
    ])
)

player_go_metrics_df

<br><br/>
#### Add Combine Metrics To Separation Data

In [ ]:
analysis_df = (
    player_go_metrics_df
    .join(
        combine_results_df, 
        left_on="wr_nfl_id", 
        right_on="nfl_id", 
        how = "inner"
    )
)

print("Final Analysis Dataset:")
print(f"    - Number of WRs: {analysis_df.select("wr_nfl_id").n_unique()}")
analysis_df

<br><br/>
## Examine Associations
***

We've previously shown that there is a statistically significant association between *accel_score* and *avg_go_separation_generated*. We will now see if we can find a statistically significant relationship between *avg_go_separation_generated* and our general Combine results.

<br><br/>
#### General Combine Results vs Separation Generated By Burst

In [ ]:
predictor = "ngs_athleticism_score" 
outcome = "avg_go_separation_generated"

***Scatter Plot***

In [ ]:
fig = px.scatter(
    data_frame = analysis_df, 
    x = predictor, 
    y = outcome, 
    trendline = "ols"
)

fig.show()

***Univariate Model***

In [ ]:
# ==========================================
# Univariate Model
# ==========================================

regression_df = analysis_df

regression_df = regression_df.filter(
    (pl.col(predictor).is_not_null())
    & 
    (pl.col(outcome).is_not_null())

)

pd_reg = regression_df.to_pandas()

X = pd_reg[predictor]
y = pd_reg[outcome]

X = sm.add_constant(X)

model = sm.OLS(y, X).fit()
print(model.summary())

<br><br/>
## Acceleration vs NGS Athleticism
***

I want to check if our `accel_score` metric is strongly correlated with `ngs_athleticism_score`. I'm checking because I want to know if our new metric adds new and valuable information that coaches wouldn't have been able to know otherwise. 

<br><br/>
#### Examine Correlation

We see below that the correlation between `accel_score` and `ngs_athleticism_score` is low. Since they are not highly correlated, our new metric likely provides new, valuable information.

In [ ]:
# =============================================
# Correlation
# =============================================

print(f"Correlation Between accel_score And ngs_athleticism_score: {round(analysis_df.select(pl.corr("accel_score", "ngs_athleticism_score")).item(), 3)}")

<br><br/>
#### What Does Our Metric Add Beyond `ngs_atleticism_score`?

Does `accel_score` add useful information beyond what is already known by `ngs_athleticism_score`? We will check this by examining how much information is added to a model that includes `ngs_athleticism_score` when `accel_score` is added.

We see below that `accel_score` appears to add valuable information. While `accel_score` has a p-value slightly above 0.10, it adds a large amount to R-squared. AIC also decreases a good bit when adding `accel_score`. Also, with such a small sample size, our relatively small p-value for `accel_score`, while adjusting for `ngs_athleticism_score` should not be ignored.

In [ ]:
# ==========================================
# NGS Athleticism Alone
# ==========================================

predictor = "ngs_athleticism_score"

regression_df = analysis_df

regression_df = regression_df.filter(
    (pl.col(predictor).is_not_null())
    & 
    (pl.col(outcome).is_not_null())

)

pd_reg = regression_df.to_pandas()

X = pd_reg[predictor]
y = pd_reg[outcome]

X = sm.add_constant(X)

model = sm.OLS(y, X).fit()
print(model.summary())

In [ ]:
# ==========================================
# Acceleration Score Alone
# ==========================================

predictor = "accel_score"

regression_df = analysis_df

regression_df = regression_df.filter(
    (pl.col(predictor).is_not_null())
    & 
    (pl.col(outcome).is_not_null())

)

pd_reg = regression_df.to_pandas()

X = pd_reg[predictor]
y = pd_reg[outcome]

X = sm.add_constant(X)

model = sm.OLS(y, X).fit()
print(model.summary())

In [ ]:
# ===========================================
# NGS Athleticism Score And Accel Score
# ===========================================

predictors = [
    "ngs_athleticism_score",
    "accel_score", 
]

corr = round(analysis_df.select(pl.corr(predictors[0], predictors[1])).item(), 2)
vif = 1 / (1 - (corr**2))
print(f"Correlation Between {predictors[0]} and {predictors[1]}: {corr} (VIF = {round(vif, 2)})\n")

regression_df = analysis_df.filter(
    pl.all_horizontal(
        [pl.col(col).is_not_null() for col in predictors + [outcome]]
    )
)

pd_reg = regression_df.to_pandas()

X = pd_reg[predictors]
y = pd_reg[outcome]

X = sm.add_constant(X)

model = sm.OLS(y, X).fit()

print(model.summary())